# Notes
[2026-06-05] - 
Created the loading in mechanism
[2026-06-08] - 
Created the first "working" model (extremely slow, though that may be due to its genormeous size)

In [117]:
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch import nn
from Bio import SeqIO 
import torch
import random
from dataclasses import dataclass

random.seed(42)
np.random.seed (42)

print ("imported")

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu" # when running locally with newer PyTorch versions with torch.accelerator support

# device = "cuda" if torch.cuda.is_available() else "cpu" # alternative for older PyTorch versions without torch.accelerator, for use on the cluster 

print(f"Using: {device} \n")

@dataclass
class ModelArgs:
    length_cutoff: int = 200
    masking_rate: float = 0.15
    batch_size=32
    src_vocab_size = 22
    tgt_vocab_size = 22
    d_model = 128
    num_heads = 4
    num_layers = 4
    d_ff = 4*d_model
    max_seq_length = 1000
    dropout = 0.1
    num_epochs=100
    learning_rate = 1e-3
    weight_decay = 1e-3




imported
Using: mps 



My plan is to separate the file into a 

In [118]:
data_path = "./data/uniprotkb_taxonomy_id_1002366_2026_06_05.fasta"

sequences = [str(record.seq) for record in SeqIO.parse(data_path, "fasta")]
sequences = [seq for seq in sequences if len(seq) < ModelArgs.length_cutoff]

print("Number of sequences:", len(sequences))

Number of sequences: 1073


In [119]:
AminoAcids = sorted(list(set("".join(sequences))))  # X is a padding, - is masking
aa_vocab= ["-"] + ["X"]+ AminoAcids 
print(len(aa_vocab))
print(aa_vocab)

aa_stoi = {s: i for i, s in enumerate(aa_vocab)}
aa_itos = {i: s for i, s in enumerate(aa_vocab)}

22
['-', 'X', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']


In [120]:
# def collate_sequences(seq):
#     masked_seq = "".join("-" if torch.rand(1).item() < ModelArgs.masking_rate else aa for aa in seq)
#     return masked_seq, seq

# dataset = [collate_sequences(seq) for seq in sequences]

In [121]:
traindata, valdata, testdata = torch.utils.data.random_split(sequences, [0.80, 0.10, 0.10])

print(f"Train size: {len(traindata)}, Val size: {len(valdata)}, Test size: {len(testdata)} \n")


## Should i sort them by length to reduce the amound of padding?

Train size: 859, Val size: 107, Test size: 107 



In [122]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch
import torch.nn.functional as F


In [123]:
# def collate_fn(batch):
#     original = batch
#     masked = [
#         "".join("-" if torch.rand(1).item() < ModelArgs.masking_rate else aa for aa in seq)
#         for seq in original
#     ]

#     x = [torch.tensor([aa_stoi[aa] for aa in seq]) for seq in masked]

#     y = [
#         torch.tensor([aa_stoi[aa] if m == "-" else -100 for aa, m in zip(orig, mask)])
#         for orig, mask in zip(original, masked)
#     ]

#     lengths = torch.tensor([len(seq) for seq in x])

#     x = pad_sequence(x, batch_first=True, padding_value=aa_stoi["-"])
#     y = pad_sequence(y, batch_first=True, padding_value=-100)

#     x = F.one_hot(x, num_classes=len(aa_stoi)).float()

#     attention_mask = torch.arange(x.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)

#     x[~attention_mask] = 0.0

#     return x, y, attention_mask

In [124]:
def make_masked(seq, mask_prob=0.15):
    """
    BERT-style masking.

    15% of positions are selected for prediction.
    Of selected positions:
      80% become "-"
      10% become random amino acid
      10% stay unchanged.

    Labels are -100 for positions not selected for prediction.
    """
    masked = []
    labels = []

    for aa in seq:
        if torch.rand(1).item() < mask_prob:
            labels.append(aa_stoi[aa])

            r = torch.rand(1).item()

            if r < 0.8:
                masked.append("-")
            elif r < 0.9:
                masked.append(random.choice(list(AminoAcids)))
            else:
                masked.append(aa)
        else:
            masked.append(aa)
            labels.append(-100)

    return "".join(masked), torch.tensor(labels, dtype=torch.long)


def make_collate_fn(mask_rate):
    def collate_fn(batch):
        original = list(batch)

        masked_data = [
            make_masked(seq, mask_rate)
            for seq in original
        ]

        masked = [item[0] for item in masked_data]
        y = [item[1] for item in masked_data]

        x_ids = [
            torch.tensor([aa_stoi[aa] for aa in seq], dtype=torch.long)
            for seq in masked
        ]

        lengths = torch.tensor([len(seq) for seq in x_ids], dtype=torch.long)

        x_ids = pad_sequence(
            x_ids,
            batch_first=True,
            padding_value=aa_stoi["X"]
        )

        y = pad_sequence(
            y,
            batch_first=True,
            padding_value=-100
        )

        attention_mask = torch.arange(x_ids.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)

        return x_ids, y, attention_mask

    return collate_fn


In [125]:
train_loader = DataLoader(traindata, batch_size=ModelArgs.batch_size, shuffle=True, collate_fn=make_collate_fn(ModelArgs.masking_rate))
val_loader = DataLoader(valdata, batch_size=ModelArgs.batch_size, shuffle=False, collate_fn=make_collate_fn(ModelArgs.masking_rate))
test_loader = DataLoader(testdata, batch_size=ModelArgs.batch_size, shuffle=False, collate_fn=make_collate_fn(ModelArgs.masking_rate))

In [126]:
# x_batch, y_batch, attention_mask = next(iter(train_loader))
# print(x_batch.shape)
# print(y_batch.shape)
# print(attention_mask.shape)

In [127]:
import torch
import torch.nn as nn
import math
import copy

# Encoder-only architecture
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output
    
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.GELU = nn.GELU() # changed from ReLU

    def forward(self, x):
        return self.fc2(self.GELU(self.fc1(x)))
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class LocalConv(nn.Module):
    def __init__(self, d_model, kernel_size=7, dropout=0.1):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=kernel_size, padding=padding, groups=d_model),
            nn.GELU(),
            nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=1),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        return x

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.local_convolution = LocalConv(d_model, kernel_size=7, dropout=dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        conv_output = self.local_convolution(x)
        x = self.norm2(x + self.dropout(conv_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x


class Transformer(nn.Module):
    def __init__(self, src_vocab_size,tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=1)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.final_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)


    def forward(self, src, attention_mask=None):
        x = self.encoder_embedding(src)
        x = self.positional_encoding(x)
        x = self.dropout(x)

        if attention_mask is not None:
            x = x * attention_mask.unsqueeze(-1)
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
        else:
            mask = None

        for enc_layer in self.encoder_layers:
            x = enc_layer(x, mask=mask)

            if attention_mask is not None:
                x = x * attention_mask.unsqueeze(-1)

        x = self.final_norm(x)
        output = self.fc(x)

        return output

transformer = Transformer(ModelArgs.src_vocab_size, ModelArgs.tgt_vocab_size, ModelArgs.d_model, ModelArgs.num_heads, ModelArgs.num_layers, ModelArgs.d_ff, ModelArgs.max_seq_length, ModelArgs.dropout)

print(transformer)

Transformer(
  (encoder_embedding): Embedding(22, 128, padding_idx=1)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-3): 4 x EncoderLayer(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=128, out_features=128, bias=True)
        (W_k): Linear(in_features=128, out_features=128, bias=True)
        (W_v): Linear(in_features=128, out_features=128, bias=True)
        (W_o): Linear(in_features=128, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (local_convolution): LocalConv(
        (conv): Sequential(
          (0): Conv1d(128, 128, kernel_size=(7,), stride=(1,), padding=(3,), groups=128)
          (1): GELU(approximate='none')
          (2): Conv1d(128, 128, kernel_size=(1,), stride=(1,))
          (3): Dropout(p=0.1, inplace=False)
        )
      )
      (norm3): LayerNor

In [128]:
import torch
import torch.nn as nn
import torch.optim as optim

def run_epoch(model, loader, criterion, optimizer, device, tgt_vocab_size, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0

    for X_batch, y_batch, attention_mask in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        attention_mask = attention_mask.to(device).bool() # can we not make it boolian from the get go?

        output = model(X_batch, attention_mask)

        loss = criterion(
            output.contiguous().view(-1, tgt_vocab_size),
            y_batch.contiguous().view(-1)
        )

        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def train_model(
    model,
    train_loader,
    val_loader,
    device,
    tgt_vocab_size,
    num_epochs=100,
    learning_rate=1e-4,
    save_path="./outputs/weights.pth"
):
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=ModelArgs.weight_decay
    )

    model = model.to(device)

    for epoch in range(num_epochs):
        train_loss = run_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            tgt_vocab_size=tgt_vocab_size,
            training=True
        )

        with torch.no_grad():
            val_loss = run_epoch(
                model=model,
                loader=val_loader,
                criterion=criterion,
                optimizer=optimizer,
                device=device,
                tgt_vocab_size=tgt_vocab_size,
                training=False
            )

        print(
            f"Epoch: {epoch + 1}, "
            f"Train Loss: {train_loss:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Difference: {val_loss-train_loss}"
        )

    torch.save(model.state_dict(), save_path)

    return model

criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

transformer = train_model(
    model=transformer,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    tgt_vocab_size=ModelArgs.tgt_vocab_size,
    num_epochs=ModelArgs.num_epochs,
    learning_rate=ModelArgs.learning_rate,
    save_path="./outputs/weights.pth"
    # save_path=save_path
)

Epoch: 1, Train Loss: 2.8157, Val Loss: 2.7142, Difference: -0.10152714340775093
Epoch: 2, Train Loss: 2.7134, Val Loss: 2.7169, Difference: 0.003466727557005722
Epoch: 3, Train Loss: 2.6916, Val Loss: 2.6730, Difference: -0.018518070379892837
Epoch: 4, Train Loss: 2.6816, Val Loss: 2.6962, Difference: 0.01453032979258806
Epoch: 5, Train Loss: 2.6769, Val Loss: 2.6280, Difference: -0.048917183169612244
Epoch: 6, Train Loss: 2.6692, Val Loss: 2.6668, Difference: -0.002359814114040848
Epoch: 7, Train Loss: 2.6600, Val Loss: 2.6476, Difference: -0.012482987509833343
Epoch: 8, Train Loss: 2.6470, Val Loss: 2.6539, Difference: 0.006932675838470459
Epoch: 9, Train Loss: 2.6559, Val Loss: 2.6544, Difference: -0.0015146975164062049
Epoch: 10, Train Loss: 2.6491, Val Loss: 2.6633, Difference: 0.014132259068665665
Epoch: 11, Train Loss: 2.6443, Val Loss: 2.6527, Difference: 0.008389620869248215
Epoch: 12, Train Loss: 2.6439, Val Loss: 2.6382, Difference: -0.0056942215672246554
Epoch: 13, Train L

In [136]:
def test_model(model, test_loader, device, tgt_vocab_size, criterion):
    model.eval()

    total_correct = 0
    total_positions = 0
    total_test_loss = 0

    with torch.no_grad():
        for X_batch, y_batch, attention_mask in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            attention_mask = attention_mask.to(device).bool()

            output = model(X_batch, attention_mask)

            loss = criterion(
                output.contiguous().view(-1, tgt_vocab_size),
                y_batch.contiguous().view(-1)
            )

            total_test_loss += loss.item()

            predicted = output.argmax(dim=-1)

            mask = y_batch != -100

            total_correct += (predicted[mask] == y_batch[mask]).sum().item()
            total_positions += mask.sum().item()

    avg_test_loss = total_test_loss / len(test_loader)
    test_accuracy = total_correct / total_positions * 100

    print(f"Test Loss: {avg_test_loss:.4f}")
    print(f"Masked Test Accuracy: {test_accuracy:.2f}%")

    return avg_test_loss, test_accuracy

test_loss, test_accuracy = test_model(
    model=transformer,
    test_loader=test_loader,
    device=device,
    tgt_vocab_size=ModelArgs.tgt_vocab_size,
    criterion=criterion
)

Test Loss: 2.6209
Masked Test Accuracy: 21.56%
